> **Versión pública:** se han retirado salidas de ejecución y celdas con identificadores de registros específicos. El notebook requiere archivos de datos privados autorizados para reproducir el entrenamiento; esos archivos no se distribuyen en este repositorio. Las métricas agregadas se encuentran en `results/semana5/`.


# UEES · Proyecto Integrador Final — Semana 3
## Análisis de sobreajuste y subajuste
**Autor:** Erik R. Nevárez  
**Estado:** experimento ejecutado; resultados exploratorios.

**Proyecto:** automatización de borradores de informes radiológicos mediante reconocimiento de voz y procesamiento de lenguaje natural. **Piloto:** radiografías (RX).

**Alcance del notebook:** clasificador auxiliar de coincidencias de versos en informes históricos. **No** mide todavía la identificación de versos a partir del dictado.


## 1. Alcance y fuentes
- Se exploraron cinco audios con Whisper Base (CPU) y Small (GPU T4), como antecedente; sus tiempos no son curvas de aprendizaje.
- Se emplearon informes históricos y versos configurados. Se verificaron manualmente tres ejemplos de dictado, insuficientes para entrenar un clasificador robusto de voz.
- Para el experimento auxiliar se construyeron etiquetas aproximadas por coincidencia exacta de versos en el informe final. La ausencia de coincidencia **no** significa ausencia clínica de verso.
- Los textos completos de los versos se retiraron de la entrada antes de vectorizar; pueden subsistir pistas indirectas y las etiquetas no son verdad de terreno clínica.
- No se deben compartir datos personales de pacientes ni audios identificables en la entrega.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tiempos observados y comunicados durante las ejecuciones reales de Colab.
# Base se ejecutó en CPU; Small, en GPU T4. No aislan efecto del modelo.
tiempos = pd.DataFrame({
    "CPT": ["72100", "70486", "890001", "73090", "73120"],
    "base_cpu_s": [24.1, 25.6, 7.6, 6.3, 2.8],
    "small_gpu_s": [4.6, 7.0, 1.2, 2.7, 0.6],
})
display(tiempos)
print("Total Base/CPU:", round(tiempos.base_cpu_s.sum(), 1), "s")
print("Total Small/GPU:", round(tiempos.small_gpu_s.sum(), 1), "s")

In [ ]:
ax = tiempos.set_index("CPT")[['base_cpu_s', 'small_gpu_s']].plot(kind='bar', figsize=(9,4))
ax.set(title='Tiempo de transcripción observado por CPT (modelo y hardware distintos)', xlabel='CPT', ylabel='Segundos')
ax.legend(['Whisper Base / CPU', 'Whisper Small / GPU T4'])
ax.grid(axis='y', alpha=.3)
plt.tight_layout()
plt.savefig('comparacion_tiempos_asr.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Definición del experimento supervisado
**Objetivo experimental:** predecir si el informe final contiene una coincidencia exacta con un verso configurado, usando texto residual tras retirar los versos.

**Objetivo futuro del proyecto:** detectar en transcripciones de audio expresiones breves dictadas por la radióloga y seleccionar el verso correcto según CPT. Ese objetivo requiere más dictados con etiquetas verificadas y evaluación independiente; no debe confundirse con el presente experimento.

**Control metodológico:** agrupación por orden médica al dividir entrenamiento/validación, vectorizador TF-IDF ajustado solo en entrenamiento, y validación sin remuestreo. Las métricas reflejan únicamente la tarea auxiliar y una sola partición de datos.


In [ ]:
from google.colab import files
import pandas as pd

archivos = files.upload()

historico = pd.read_excel("EjemploData(1).xlsx")

print("Total de registros:", len(historico))
print("Columnas disponibles:")
print(historico.columns.tolist())

In [ ]:
resumen_cpt = (
    historico.groupby(["CPT", "NOMBRE_PRESTACION_MEDICA"])
    .size()
    .reset_index(name="cantidad")
    .sort_values("cantidad", ascending=False)
)

print("Total de CPT diferentes:", historico["CPT"].nunique())

display(resumen_cpt.head(30))

In [ ]:
import pandas as pd

cpt_objetivo = ["72100", "73120"]

rx = historico[
    historico["CPT"].astype(str).str.strip().isin(cpt_objetivo)
].copy()

rx["informe"] = (
    rx["VALOR_RESULTADO"].fillna("").astype(str)
    + " "
    + rx["VALOR_RESULTADO_PART2"].fillna("").astype(str)
)

rx["posible_cambio_leve"] = rx["informe"].str.contains(
    r"cambios leves",
    case=False,
    regex=True
)

print("Registros de los CPT seleccionados:", len(rx))
print("Registros que contienen 'cambios leves':",
      rx["posible_cambio_leve"].sum())

print("\nResultados por CPT:")
display(
    rx.groupby("CPT")["posible_cambio_leve"]
    .agg(["count", "sum"])
    .rename(columns={
        "count": "total_registros",
        "sum": "coincidencias"
    })
)

In [ ]:
frase = "Cambios osteodegenerativos leves"

coincidencias = rx[
    rx["informe"].str.contains(
        frase,
        case=False,
        regex=False,
        na=False
    )
]

print("Registros con la conclusión:", len(coincidencias))

print("\nCoincidencias por CPT:")
display(coincidencias.groupby("CPT").size())

In [ ]:
print("Registros encontrados:", len(coincidencias))

print(
    "Órdenes diferentes:",
    coincidencias["SECUENCIA_ORDEN"].nunique()
)

print(
    "Audios diferentes:",
    coincidencias["NOMBRE_ARCHIVO_AUDIO"].nunique()
)

print("\nDistribución de registros por orden:")
display(
    coincidencias.groupby("SECUENCIA_ORDEN")
    .size()
    .value_counts()
    .sort_index()
)

In [ ]:
from google.colab import files
import pandas as pd

files.upload()

versos = pd.read_excel("VersosPatologicosDraMoncayo.xlsx")

print("Versos configurados:", len(versos))

display(
    versos[
        versos["CODIGO_CPT"].astype(str).isin(
            ["72100", "73120", "73090"]
        )
    ][
        ["CODIGO_CPT", "NOMBRE_VERSO", "TEXTO_VERSO"]
    ]
)

In [ ]:
import re
import pandas as pd

def normalizar(texto):
    texto = str(texto) if pd.notna(texto) else ""
    texto = texto.replace("_x000D_", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip().lower()

cpts = ["73090", "73120", "72100"]

versos_rx = versos[
    versos["CODIGO_CPT"].astype(str).str.strip().isin(cpts)
].copy()

historico_rx = historico[
    historico["CPT"].astype(str).str.strip().isin(cpts)
].copy()

historico_rx["texto_normalizado"] = (
    historico_rx["VALOR_RESULTADO"].fillna("").astype(str)
    + " "
    + historico_rx["VALOR_RESULTADO_PART2"].fillna("").astype(str)
).apply(normalizar)

resultados = []

for _, verso in versos_rx.iterrows():
    cpt = str(verso["CODIGO_CPT"]).strip()
    texto_verso = normalizar(verso["TEXTO_VERSO"])

    if not texto_verso:
        continue

    registros = historico_rx[
        historico_rx["CPT"].astype(str).str.strip() == cpt
    ]

    encontrados = registros[
        registros["texto_normalizado"].str.contains(
            texto_verso, regex=False, na=False
        )
    ]

    resultados.append({
        "CPT": cpt,
        "VERSO": verso["NOMBRE_VERSO"],
        "COINCIDENCIAS": len(encontrados),
        "ORDENES_UNICAS": encontrados["SECUENCIA_ORDEN"].nunique()
    })

resumen = pd.DataFrame(resultados)

display(
    resumen.sort_values(
        "COINCIDENCIAS", ascending=False
    )
)

In [ ]:
# Identificar órdenes con coincidencias de versos
ordenes_con_verso = set()

for _, verso in versos_rx.iterrows():
    cpt = str(verso["CODIGO_CPT"]).strip()
    texto_verso = normalizar(verso["TEXTO_VERSO"])

    if not texto_verso:
        continue

    encontrados = historico_rx[
        (historico_rx["CPT"].astype(str).str.strip() == cpt)
        & (
            historico_rx["texto_normalizado"].str.contains(
                texto_verso, regex=False, na=False
            )
        )
    ]

    ordenes_con_verso.update(
        encontrados["SECUENCIA_ORDEN"].dropna().astype(str)
    )

total_ordenes = historico_rx["SECUENCIA_ORDEN"].nunique()

print("Total de órdenes de los 3 CPT:", total_ordenes)
print("Órdenes con coincidencia de verso:", len(ordenes_con_verso))
print(
    "Órdenes sin coincidencia:",
    total_ordenes - len(ordenes_con_verso)
)

In [ ]:
print("Distribución de órdenes por verso:")
display(
    resumen[
        resumen["ORDENES_UNICAS"] > 0
    ][["CPT", "VERSO", "ORDENES_UNICAS"]]
    .sort_values("ORDENES_UNICAS", ascending=False)
)

print(
    "\nTotal de clases con coincidencias:",
    (resumen["ORDENES_UNICAS"] > 0).sum()
)

In [ ]:
# Revisar cómo se distribuyen las filas por orden médica
estructura = (
    historico_rx
    .groupby(["CPT", "SECUENCIA_ORDEN"])
    .agg(
        filas=("VALOR_RESULTADO", "size"),
        audios=("NOMBRE_ARCHIVO_AUDIO", "nunique")
    )
    .reset_index()
)

print("Total de órdenes:", len(estructura))

print("\nCantidad de filas por orden:")
display(
    estructura["filas"]
    .value_counts()
    .sort_index()
)

print("\nCantidad de audios por orden:")
display(
    estructura["audios"]
    .value_counts()
    .sort_index()
)

In [ ]:
print("Filas del histórico RX:", len(historico_rx))

print(
    "Órdenes únicas sin contar vacíos:",
    historico_rx["SECUENCIA_ORDEN"].nunique()
)

print(
    "Combinaciones CPT + orden:",
    historico_rx.groupby(
        ["CPT", "SECUENCIA_ORDEN"]
    ).ngroups
)

print(
    "Órdenes asociadas a más de un CPT:",
    (
        historico_rx.groupby("SECUENCIA_ORDEN")["CPT"]
        .nunique() > 1
    ).sum()
)

In [ ]:
estudios = (
    historico_rx
    .groupby(["SECUENCIA_ORDEN", "CPT"])
    .agg(
        filas=("VALOR_RESULTADO", "size"),
        audios=("NOMBRE_ARCHIVO_AUDIO", "nunique")
    )
    .reset_index()
)

print("Total de estudios:", len(estudios))

print("\nEstudios por CPT:")
display(
    estudios.groupby("CPT")
    .size()
    .reset_index(name="estudios")
)

In [ ]:
# Agrupar las partes del informe por estudio
def unir_textos(serie):
    textos = [
        normalizar(t)
        for t in serie
        if pd.notna(t) and str(t).strip()
    ]
    return " ".join(textos)

historico_rx["texto_completo"] = (
    historico_rx["VALOR_RESULTADO"].fillna("").astype(str)
    + " "
    + historico_rx["VALOR_RESULTADO_PART2"].fillna("").astype(str)
)

dataset_estudios = (
    historico_rx
    .groupby(["SECUENCIA_ORDEN", "CPT"], as_index=False)
    .agg(
        informe=("texto_completo", unir_textos),
        audio=("NOMBRE_ARCHIVO_AUDIO", "first")
    )
)

print("Estudios preparados:", len(dataset_estudios))
print("Informes vacíos:",
      (dataset_estudios["informe"].str.strip() == "").sum())

display(
    dataset_estudios[["CPT", "SECUENCIA_ORDEN", "audio"]]
    .head(10)
)

In [ ]:
# Etiquetado experimental por coincidencia textual
dataset_estudios["etiqueta_experimental"] = "SIN_COINCIDENCIA"

for _, verso in versos_rx.iterrows():
    cpt = str(verso["CODIGO_CPT"]).strip()
    texto_verso = normalizar(verso["TEXTO_VERSO"])

    if not texto_verso:
        continue

    condicion = (
        (dataset_estudios["CPT"].astype(str) == cpt)
        & dataset_estudios["informe"].str.contains(
            texto_verso, regex=False, na=False
        )
    )

    dataset_estudios.loc[
        condicion, "etiqueta_experimental"
    ] = str(verso["NOMBRE_VERSO"])

print("Total de estudios:", len(dataset_estudios))

print("\nDistribución de etiquetas experimentales:")
display(
    dataset_estudios["etiqueta_experimental"]
    .value_counts()
)

In [ ]:
# Contar coincidencias de versos por estudio
coincidencias = []

for _, estudio in dataset_estudios.iterrows():
    cantidad = 0

    for _, verso in versos_rx.iterrows():
        if str(estudio["CPT"]) != str(verso["CODIGO_CPT"]):
            continue

        texto_verso = normalizar(verso["TEXTO_VERSO"])

        if texto_verso and texto_verso in estudio["informe"]:
            cantidad += 1

    coincidencias.append(cantidad)

dataset_estudios["cantidad_versos"] = coincidencias

print("Cantidad de versos detectados por estudio:")

display(
    dataset_estudios["cantidad_versos"]
    .value_counts()
    .sort_index()
)

print(
    "Estudios con más de un verso:",
    (dataset_estudios["cantidad_versos"] > 1).sum()
)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Variable que intentaremos predecir
dataset_estudios["objetivo"] = (
    dataset_estudios["cantidad_versos"] > 0
).astype(int)

# Separación por orden médica para evitar filtraciones
separador = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

indice_train, indice_val = next(
    separador.split(
        dataset_estudios,
        dataset_estudios["objetivo"],
        groups=dataset_estudios["SECUENCIA_ORDEN"]
    )
)

train = dataset_estudios.iloc[indice_train].copy()
val = dataset_estudios.iloc[indice_val].copy()

print("Estudios de entrenamiento:", len(train))
print("Estudios de validación:", len(val))

print("\nDistribución de clases en entrenamiento:")
print(train["objetivo"].value_counts())

print("\nDistribución de clases en validación:")
print(val["objetivo"].value_counts())

print(
    "\nÓrdenes compartidas entre conjuntos:",
    len(
        set(train["SECUENCIA_ORDEN"])
        & set(val["SECUENCIA_ORDEN"])
    )
)

In [ ]:
# Crear una versión del informe sin los versos configurados
def quitar_versos(informe, cpt):
    texto = informe

    versos_cpt = versos_rx[
        versos_rx["CODIGO_CPT"].astype(str) == str(cpt)
    ]

    for _, verso in versos_cpt.iterrows():
        frase = normalizar(verso["TEXTO_VERSO"])

        if frase:
            texto = texto.replace(frase, " ")

    return " ".join(texto.split())


for conjunto in [train, val]:
    conjunto["texto_sin_versos"] = conjunto.apply(
        lambda fila: quitar_versos(
            fila["informe"],
            fila["CPT"]
        ),
        axis=1
    )

print("Entrenamiento:")
print("Informes vacíos después de quitar versos:",
      (train["texto_sin_versos"].str.len() == 0).sum())

print("\nValidación:")
print("Informes vacíos después de quitar versos:",
      (val["texto_sin_versos"].str.len() == 0).sum())

print("\nLongitud promedio del texto restante:")
print("Entrenamiento:",
      round(train["texto_sin_versos"].str.len().mean(), 1))
print("Validación:",
      round(val["texto_sin_versos"].str.len().mean(), 1))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, log_loss

# Convertir textos en características numéricas
vectorizador = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    ngram_range=(1, 2)
)

X_train = vectorizador.fit_transform(
    train["texto_sin_versos"]
).toarray()

X_val = vectorizador.transform(
    val["texto_sin_versos"]
).toarray()

y_train = train["objetivo"].to_numpy()
y_val = val["objetivo"].to_numpy()

# Modelo inicial
modelo_base = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    alpha=0.0001,
    max_iter=1,
    warm_start=True,
    random_state=42
)

historial = []

for epoca in range(1, 21):
    modelo_base.fit(X_train, y_train)

    prob_train = modelo_base.predict_proba(X_train)
    prob_val = modelo_base.predict_proba(X_val)

    historial.append({
        "epoca": epoca,
        "loss_train": log_loss(y_train, prob_train),
        "loss_val": log_loss(y_val, prob_val),
        "accuracy_train": accuracy_score(
            y_train, modelo_base.predict(X_train)
        ),
        "accuracy_val": accuracy_score(
            y_val, modelo_base.predict(X_val)
        )
    })

historial_base = pd.DataFrame(historial)

display(historial_base.tail())

# Curvas de pérdida
plt.figure(figsize=(9, 5))
plt.plot(
    historial_base["epoca"],
    historial_base["loss_train"],
    label="Entrenamiento"
)
plt.plot(
    historial_base["epoca"],
    historial_base["loss_val"],
    label="Validación"
)

plt.xlabel("Épocas")
plt.ylabel("Pérdida (Log Loss)")
plt.title("Modelo inicial: entrenamiento vs. validación")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score
)

predicciones = modelo_base.predict(X_val)

print("REPORTE DE CLASIFICACIÓN")
print(
    classification_report(
        y_val,
        predicciones,
        labels=[0, 1],
        target_names=[
            "Sin coincidencia",
            "Con verso detectado"
        ],
        zero_division=0
    )
)

print(
    "Exactitud balanceada:",
    round(balanced_accuracy_score(y_val, predicciones), 4)
)

ConfusionMatrixDisplay.from_predictions(
    y_val,
    predicciones,
    labels=[0, 1],
    display_labels=["Sin coincidencia", "Con verso"],
    cmap="Blues",
    values_format="d"
)

plt.title("Matriz de confusión - Modelo inicial")
plt.show()

In [ ]:
from sklearn.utils import resample

# Identificar ejemplos de cada clase
indices_0 = np.where(y_train == 0)[0]
indices_1 = np.where(y_train == 1)[0]

# Repetir ejemplos de la clase minoritaria
indices_1_balanceados = resample(
    indices_1,
    replace=True,
    n_samples=len(indices_0),
    random_state=42
)

# Construir conjunto equilibrado
indices_balanceados = np.concatenate([
    indices_0,
    indices_1_balanceados
])

rng = np.random.default_rng(42)
rng.shuffle(indices_balanceados)

X_train_balanceado = X_train[indices_balanceados]
y_train_balanceado = y_train[indices_balanceados]

print("Distribución original:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nDistribución balanceada:")
print(
    pd.Series(y_train_balanceado)
    .value_counts()
    .sort_index()
)

print("\nEstudios de validación:", len(y_val))

In [ ]:
# Modelo 2: entrenamiento con clases balanceadas
modelo_balanceado = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    alpha=0.0001,
    max_iter=1,
    warm_start=True,
    random_state=42
)

historial_2 = []

for epoca in range(1, 21):
    modelo_balanceado.fit(
        X_train_balanceado,
        y_train_balanceado
    )

    prob_train = modelo_balanceado.predict_proba(
        X_train_balanceado
    )
    prob_val = modelo_balanceado.predict_proba(X_val)

    historial_2.append({
        "epoca": epoca,
        "loss_train": log_loss(
            y_train_balanceado, prob_train
        ),
        "loss_val": log_loss(y_val, prob_val),
        "accuracy_train": accuracy_score(
            y_train_balanceado,
            modelo_balanceado.predict(X_train_balanceado)
        ),
        "accuracy_val": accuracy_score(
            y_val, modelo_balanceado.predict(X_val)
        )
    })

historial_balanceado = pd.DataFrame(historial_2)

# Comparación de las curvas de validación
plt.figure(figsize=(9, 5))

plt.plot(
    historial_base["epoca"],
    historial_base["loss_val"],
    label="Modelo inicial"
)

plt.plot(
    historial_balanceado["epoca"],
    historial_balanceado["loss_val"],
    label="Modelo balanceado"
)

plt.xlabel("Épocas")
plt.ylabel("Pérdida de validación")
plt.title("Comparación de modelos")
plt.legend()
plt.grid(True)
plt.show()

# Evaluación del segundo modelo
pred_balanceado = modelo_balanceado.predict(X_val)

print("RESULTADOS DEL MODELO BALANCEADO")
print(
    classification_report(
        y_val,
        pred_balanceado,
        labels=[0, 1],
        target_names=[
            "Sin coincidencia",
            "Con verso detectado"
        ],
        zero_division=0
    )
)

print(
    "Exactitud balanceada:",
    round(
        balanced_accuracy_score(
            y_val, pred_balanceado
        ),
        4
    )
)

In [ ]:
# Modelo 3: balanceo + regularización L2
modelo_regularizado = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    alpha=0.1,  # Mayor regularización
    max_iter=1,
    warm_start=True,
    random_state=42
)

historial_3 = []

for epoca in range(1, 21):
    modelo_regularizado.fit(
        X_train_balanceado,
        y_train_balanceado
    )

    historial_3.append({
        "epoca": epoca,
        "loss_train": log_loss(
            y_train_balanceado,
            modelo_regularizado.predict_proba(
                X_train_balanceado
            )
        ),
        "loss_val": log_loss(
            y_val,
            modelo_regularizado.predict_proba(X_val)
        )
    })

historial_regularizado = pd.DataFrame(historial_3)

# Comparar las tres curvas de validación
plt.figure(figsize=(9, 5))

plt.plot(
    historial_base["epoca"],
    historial_base["loss_val"],
    label="Modelo inicial"
)

plt.plot(
    historial_balanceado["epoca"],
    historial_balanceado["loss_val"],
    label="Modelo balanceado"
)

plt.plot(
    historial_regularizado["epoca"],
    historial_regularizado["loss_val"],
    label="Balanceado + regularización"
)

plt.xlabel("Épocas")
plt.ylabel("Pérdida de validación")
plt.title("Comparación de los tres modelos")
plt.legend()
plt.grid(True)
plt.show()

# Evaluación del tercer modelo
pred_regularizado = modelo_regularizado.predict(X_val)

print("RESULTADOS DEL MODELO REGULARIZADO")

print(
    classification_report(
        y_val,
        pred_regularizado,
        labels=[0, 1],
        target_names=[
            "Sin coincidencia",
            "Con verso detectado"
        ],
        zero_division=0
    )
)

print(
    "Exactitud balanceada:",
    round(
        balanced_accuracy_score(
            y_val,
            pred_regularizado
        ),
        4
    )
)

In [ ]:
# Comparar curvas de entrenamiento y validación
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

modelos = [
    ("Modelo inicial", historial_base),
    ("Modelo balanceado", historial_balanceado),
    ("Balanceado + L2", historial_regularizado)
]

resumen_diagnostico = []

for ax, (nombre, datos) in zip(axes, modelos):
    ax.plot(
        datos["epoca"],
        datos["loss_train"],
        label="Entrenamiento"
    )

    ax.plot(
        datos["epoca"],
        datos["loss_val"],
        label="Validación"
    )

    ax.set_title(nombre)
    ax.set_xlabel("Épocas")
    ax.set_ylabel("Pérdida")
    ax.grid(True)
    ax.legend()

    resumen_diagnostico.append({
        "Modelo": nombre,
        "Pérdida entrenamiento": round(
            datos["loss_train"].iloc[-1], 4
        ),
        "Pérdida validación": round(
            datos["loss_val"].iloc[-1], 4
        ),
        "Mejor época validación": int(
            datos.loc[
                datos["loss_val"].idxmin(), "epoca"
            ]
        )
    })

plt.tight_layout()
plt.show()

print("RESUMEN DEL DIAGNÓSTICO")
display(pd.DataFrame(resumen_diagnostico))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

modelos_finales = {
    "Inicial": modelo_base,
    "Balanceado": modelo_balanceado,
    "Balanceado + L2": modelo_regularizado
}

resultados_finales = []

for nombre, modelo in modelos_finales.items():
    pred = modelo.predict(X_val)

    resultados_finales.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_val, pred),
        "Accuracy balanceada": balanced_accuracy_score(y_val, pred),
        "Precision clase 1": precision_score(
            y_val, pred, zero_division=0
        ),
        "Recall clase 1": recall_score(
            y_val, pred, zero_division=0
        ),
        "F1 clase 1": f1_score(
            y_val, pred, zero_division=0
        )
    })

tabla_final = pd.DataFrame(resultados_finales)

display(
    tabla_final.style.format({
        "Accuracy": "{:.2%}",
        "Accuracy balanceada": "{:.2%}",
        "Precision clase 1": "{:.2%}",
        "Recall clase 1": "{:.2%}",
        "F1 clase 1": "{:.2%}"
    })
)

# Guardar resultados para el informe
tabla_final.to_csv(
    "resultados_semana3.csv",
    index=False
)

print("Resumen generado correctamente.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(
    historial_base["epoca"],
    historial_base["accuracy_train"],
    label="Modelo inicial - Entrenamiento",
    linestyle="-"
)

plt.plot(
    historial_base["epoca"],
    historial_base["accuracy_val"],
    label="Modelo inicial - Validación",
    linestyle="--"
)

plt.plot(
    historial_balanceado["epoca"],
    historial_balanceado["accuracy_train"],
    label="Modelo balanceado - Entrenamiento",
    linestyle="-"
)

plt.plot(
    historial_balanceado["epoca"],
    historial_balanceado["accuracy_val"],
    label="Modelo balanceado - Validación",
    linestyle="--"
)

plt.xlabel("Épocas")
plt.ylabel("Exactitud (Accuracy)")
plt.title("Curvas de exactitud: entrenamiento vs. validación")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    "curvas_accuracy_semana3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Gráfica generada y guardada a 300 DPI.")

In [ ]:
# Obtener las órdenes médicas de los registros de entrenamiento
grupos_train = dataset_estudios.iloc[indice_train]["SECUENCIA_ORDEN"].to_numpy()

print("Total de registros de entrenamiento:", len(grupos_train))
print("Órdenes médicas diferentes:", len(set(grupos_train)))
print("Coincide con X_train:", len(grupos_train) == len(X_train))

## Validación cruzada final sin filtración de TF-IDF
Las curvas preliminares se reemplazan por un Pipeline (TF-IDF + MLP) y GroupKFold de 3 particiones. El vectorizador se ajusta exclusivamente con el subconjunto de entrenamiento de cada partición. Estas curvas son análisis complementarios y no sustituyen la evaluación de los tres modelos en el conjunto de validación reservado.


In [ ]:
# Curva final: vectorización dentro de cada fold y órdenes agrupadas
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import learning_curve, GroupKFold
import numpy as np
import matplotlib.pyplot as plt

# TF-IDF se ajustará dentro de cada partición
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=1000,
        min_df=2,
        ngram_range=(1, 2)
    )),
    ("modelo", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        alpha=0.0001,
        max_iter=200,
        random_state=42
    ))
])

textos_train = train["texto_sin_versos"].fillna("").astype(str).to_numpy()

tamanos, scores_train, scores_val = learning_curve(
    pipeline,
    textos_train,
    y_train,
    groups=grupos_train,
    cv=GroupKFold(n_splits=3),
    train_sizes=np.linspace(0.5, 1.0, 5),
    scoring="accuracy",
    n_jobs=-1
)

plt.figure(figsize=(9, 5))
plt.plot(
    tamanos,
    scores_train.mean(axis=1),
    "o-",
    label="Entrenamiento"
)
plt.plot(
    tamanos,
    scores_val.mean(axis=1),
    "o--",
    label="Validación agrupada"
)

plt.xlabel("Cantidad de estudios de entrenamiento")
plt.ylabel("Exactitud (Accuracy)")
plt.title("Curva de aprendizaje: TF-IDF + GroupKFold")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("learning_curve_final.png", dpi=300)
plt.show()

print("Curva corregida generada correctamente.")

In [ ]:
from sklearn.model_selection import validation_curve, GroupKFold
import numpy as np
import matplotlib.pyplot as plt

valores_alpha_final = [0.0001, 0.001, 0.01, 0.1, 1.0]

scores_train_final, scores_val_final = validation_curve(
    pipeline,
    textos_train,
    y_train,
    groups=grupos_train,
    param_name="modelo__alpha",
    param_range=valores_alpha_final,
    cv=GroupKFold(n_splits=3),
    scoring="accuracy",
    n_jobs=-1
)

plt.figure(figsize=(9, 5))

plt.semilogx(
    valores_alpha_final,
    scores_train_final.mean(axis=1),
    "o-",
    label="Entrenamiento"
)

plt.semilogx(
    valores_alpha_final,
    scores_val_final.mean(axis=1),
    "o--",
    label="Validación agrupada"
)

plt.xlabel("Regularización L2 (alpha)")
plt.ylabel("Exactitud (Accuracy)")
plt.title("Curva de validación: TF-IDF + GroupKFold")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    "validation_curve_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Curva de validación final generada correctamente.")

## Análisis de resultados y conclusiones – Semana 3

### Objetivo del experimento

El objetivo fue analizar el comportamiento de una red neuronal durante el entrenamiento y la validación, identificar posibles señales de sobreajuste y evaluar dos técnicas de mejora.

Se trabajó con 272 estudios radiológicos históricos, utilizando 201 para entrenamiento y 71 para validación. La separación se realizó por orden médica para evitar compartir estudios relacionados entre ambos conjuntos.

### Resultados obtenidos

El modelo inicial alcanzó una exactitud del 81,69 %, pero identificó solamente el 23,53 % de los estudios con versos detectados. Esto evidenció dificultades para reconocer la clase minoritaria.

Como primera mejora se aplicó un balanceo mediante remuestreo de los ejemplos de entrenamiento. Con esta técnica, el recall de la clase minoritaria aumentó al 76,47 % y su F1-score pasó del 38,10 % al 70,27 %.

La segunda mejora consistió en aplicar regularización L2. Sin embargo, no se observaron cambios en las métricas de clasificación respecto al modelo balanceado.

### Diagnóstico de sobreajuste

Durante las 20 épocas evaluadas, los tres modelos mostraron una disminución de la pérdida de validación y alcanzaron su mejor valor en la última época.

Por lo tanto, no se encontró evidencia concluyente de sobreajuste dentro del período analizado. Para confirmar su comportamiento a largo plazo sería necesario ampliar el entrenamiento y realizar nuevas evaluaciones.

### Limitaciones

Las etiquetas se construyeron mediante coincidencias de versos configurados dentro de informes radiológicos históricos. Por tanto, representan etiquetas experimentales y no verificaciones directas de lo expresado en los audios.

Aunque se retiraron los textos de los versos utilizados para construir las etiquetas, pueden permanecer pistas indirectas dentro de los informes. En consecuencia, los resultados no deben interpretarse como una validación del reconocimiento automático de dictados médicos.

### Conclusión general

El experimento permitió comprender que una exactitud general elevada no garantiza un buen desempeño sobre todas las clases.

El balanceo de datos produjo la mejora más importante, mientras que la regularización L2 no generó beneficios adicionales en esta configuración.

Los resultados constituyen una primera aproximación experimental al análisis de modelos de clasificación de textos dentro del proyecto integrador. Como siguiente etapa será necesario trabajar con transcripciones verificadas y etiquetas asociadas directamente a los dictados médicos.
### Curvas adicionales de validación cruzada
Se aplicaron `learning_curve` y `validation_curve` con `GroupKFold(n_splits=3)` y un `Pipeline` que ajusta TF-IDF dentro de cada partición. La primera muestra el efecto del tamaño de entrenamiento y la segunda compara cinco valores de regularización L2. Se trata de pruebas exploratorias, no de una evaluación clínica ni de validación de dictados.

### Referencias técnicas
- Documentación de Scikit-learn: `learning_curve`, `validation_curve`, `GroupKFold`, `Pipeline`, `TfidfVectorizer` y `MLPClassifier` (https://scikit-learn.org/stable/).
- Documentación de Matplotlib: guardado de figuras con `savefig` y parámetro `dpi` (https://matplotlib.org/stable/).


# Semana 5 — Optimización de hiperparámetros

## 8. Análisis de sensibilidad

En esta etapa se evaluará el efecto de diferentes hiperparámetros sobre el rendimiento del clasificador TF-IDF + MLP.

Se utilizará validación cruzada agrupada sobre los datos de entrenamiento, manteniendo separado el conjunto de 71 estudios reservado para evaluación final.

La métrica principal será F1 macro, complementada con el recall de la clase positiva.

Se analizarán la arquitectura de la red, la regularización, la tasa de aprendizaje y el número máximo de características TF-IDF.

In [ ]:
# Verificar las variables disponibles
variables = [
    nombre for nombre in globals()
    if not nombre.startswith("_")
]

print("Variables disponibles:")
print(", ".join(sorted(variables)))

In [ ]:
print("Entrenamiento:", len(X_train), "registros")
print("Evaluación reservada:", len(X_val), "registros")
print("Grupos de entrenamiento:", len(np.unique(grupos_train)))

print("\nDistribución de clases en entrenamiento:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nDistribución de clases en evaluación:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTipo de X_train:", type(X_train).__name__)
print("Tipo de grupos_train:", type(grupos_train).__name__)

In [ ]:
cv = GroupKFold(n_splits=5)

print("Validación cruzada agrupada:")
for numero, (idx_train, idx_valid) in enumerate(
    cv.split(X_train, y_train, groups=grupos_train), start=1
):
    grupos_entrenamiento = set(grupos_train[idx_train])
    grupos_validacion = set(grupos_train[idx_valid])

    repetidos = grupos_entrenamiento.intersection(grupos_validacion)

    print(
        f"Fold {numero}: "
        f"{len(idx_train)} entrenamiento, "
        f"{len(idx_valid)} validación, "
        f"{len(repetidos)} grupos compartidos"
    )

In [ ]:
print("MODELO BASE")
print(modelo_base)

print("\nMODELO BALANCEADO")
print(modelo_balanceado)

print("\nPIPELINE ORIGINAL")
print(pipeline)

In [ ]:
# Preparar datos para la optimización de Semana 5
textos_s5 = train["texto_sin_versos"].fillna("").astype(str).to_numpy()
etiquetas_s5 = train["objetivo"].to_numpy()
grupos_s5 = train["SECUENCIA_ORDEN"].to_numpy()

print("Textos disponibles:", len(textos_s5))
print("Etiquetas disponibles:", len(etiquetas_s5))
print("Órdenes médicas:", len(np.unique(grupos_s5)))
print("Textos vacíos:", sum(len(t.strip()) == 0 for t in textos_s5))

print("Coinciden las etiquetas:",
      np.array_equal(etiquetas_s5, y_train))

print("Coinciden los grupos:",
      np.array_equal(grupos_s5, grupos_train))

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, f1_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
import pandas as pd
import numpy as np

arquitecturas = [
    (32,),
    (64, 32),
    (128, 64)
]

resultados_arquitectura = []

cv_s5 = GroupKFold(n_splits=5)

for arquitectura in arquitecturas:

    modelo_s5 = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=1000,
            min_df=2,
            ngram_range=(1, 2)
        )),
        ("modelo", MLPClassifier(
            hidden_layer_sizes=arquitectura,
            alpha=0.0001,
            learning_rate_init=0.001,
            max_iter=200,
            random_state=42
        ))
    ])

    scores = cross_validate(
        modelo_s5,
        textos_s5,
        etiquetas_s5,
        groups=grupos_s5,
        cv=cv_s5,
        scoring={
            "f1_macro": "f1_macro",
            "recall_positivo": make_scorer(
                recall_score,
                pos_label=1,
                zero_division=0
            )
        },
        n_jobs=1
    )

    resultados_arquitectura.append({
        "Arquitectura": str(arquitectura),
        "F1 macro promedio": np.mean(scores["test_f1_macro"]),
        "F1 macro desviación": np.std(scores["test_f1_macro"]),
        "Recall positivo promedio": np.mean(scores["test_recall_positivo"])
    })

tabla_arquitectura = pd.DataFrame(resultados_arquitectura)

display(
    tabla_arquitectura.sort_values(
        "F1 macro promedio",
        ascending=False
    ).round(4)
)

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
import numpy as np

for iteraciones in [200, 400, 600]:
    modelo_prueba = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=1000,
            min_df=2,
            ngram_range=(1, 2)
        )),
        ("modelo", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=0.0001,
            learning_rate_init=0.001,
            max_iter=iteraciones,
            random_state=42
        ))
    ])

    resultados_cv = cross_validate(
        modelo_prueba,
        textos_s5,
        etiquetas_s5,
        groups=grupos_s5,
        cv=cv_s5,
        scoring="f1_macro",
        return_estimator=True,
        n_jobs=1
    )

    epocas = [
        estimador.named_steps["modelo"].n_iter_
        for estimador in resultados_cv["estimator"]
    ]

    print(
        f"Máximo {iteraciones} iteraciones | "
        f"F1 macro: {np.mean(resultados_cv['test_score']):.4f} | "
        f"Iteraciones utilizadas: {epocas}"
    )

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
import numpy as np
import pandas as pd

valores_alpha_s5 = [0.0001, 0.001, 0.01, 0.1, 1.0]

resultados_alpha_s5 = []

for alpha in valores_alpha_s5:
    modelo_alpha_s5 = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=1000,
            min_df=2,
            ngram_range=(1, 2)
        )),
        ("modelo", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=alpha,
            learning_rate_init=0.001,
            max_iter=400,
            random_state=42
        ))
    ])

    scores = cross_validate(
        modelo_alpha_s5,
        textos_s5,
        etiquetas_s5,
        groups=grupos_s5,
        cv=cv_s5,
        scoring={
            "f1_macro": "f1_macro",
            "recall_positivo": "recall"
        },
        n_jobs=1
    )

    resultados_alpha_s5.append({
        "Alpha": alpha,
        "F1 macro promedio": np.mean(scores["test_f1_macro"]),
        "F1 macro desviación": np.std(scores["test_f1_macro"]),
        "Recall positivo promedio": np.mean(scores["test_recall_positivo"])
    })

tabla_alpha_s5 = pd.DataFrame(resultados_alpha_s5)

display(tabla_alpha_s5.round(4))

In [ ]:
print("Textos:", len(textos_s5))
print("Etiquetas:", len(etiquetas_s5))
print("Grupos:", len(grupos_s5))

In [ ]:
valores_lr_s5 = [0.0001, 0.001, 0.01]
resultados_lr_s5 = []

for lr in valores_lr_s5:

    modelo_lr_s5 = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=1000,
            min_df=2,
            ngram_range=(1, 2)
        )),
        ("modelo", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=0.0001,
            learning_rate_init=lr,
            max_iter=400,
            random_state=42
        ))
    ])

    scores = cross_validate(
        modelo_lr_s5,
        textos_s5,
        etiquetas_s5,
        groups=grupos_s5,
        cv=cv_s5,
        scoring={
            "f1_macro": "f1_macro",
            "recall_positivo": "recall"
        },
        return_estimator=True,
        n_jobs=1
    )

    epocas = [
        estimador.named_steps["modelo"].n_iter_
        for estimador in scores["estimator"]
    ]

    resultados_lr_s5.append({
        "Learning rate": lr,
        "F1 macro promedio": np.mean(scores["test_f1_macro"]),
        "F1 macro desviación": np.std(scores["test_f1_macro"]),
        "Recall positivo promedio": np.mean(scores["test_recall_positivo"]),
        "Folds en límite de iteraciones": sum(e >= 400 for e in epocas)
    })

tabla_lr_s5 = pd.DataFrame(resultados_lr_s5)

display(tabla_lr_s5.round(4))

In [ ]:
valores_features_s5 = [500, 1000, 2000, 3000]
resultados_features_s5 = []

for features in valores_features_s5:

    modelo_features_s5 = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=features,
            min_df=2,
            ngram_range=(1, 2)
        )),
        ("modelo", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=0.0001,
            learning_rate_init=0.001,
            max_iter=400,
            random_state=42
        ))
    ])

    scores = cross_validate(
        modelo_features_s5,
        textos_s5,
        etiquetas_s5,
        groups=grupos_s5,
        cv=cv_s5,
        scoring={
            "f1_macro": "f1_macro",
            "recall_positivo": "recall"
        },
        n_jobs=1
    )

    resultados_features_s5.append({
        "Max features": features,
        "F1 macro promedio": np.mean(scores["test_f1_macro"]),
        "F1 macro desviación": np.std(scores["test_f1_macro"]),
        "Recall positivo promedio": np.mean(
            scores["test_recall_positivo"]
        )
    })

tabla_features_s5 = pd.DataFrame(resultados_features_s5)

display(tabla_features_s5.round(4))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador_s5 = TfidfVectorizer(
    max_features=None,
    min_df=2,
    ngram_range=(1, 2)
)

vectorizador_s5.fit(textos_s5)

print(
    "Características disponibles:",
    len(vectorizador_s5.get_feature_names_out())
)

In [ ]:
resumen_sensibilidad_s5 = pd.DataFrame([
    {
        "Hiperparámetro": "Arquitectura MLP",
        "F1 mínimo": tabla_arquitectura["F1 macro promedio"].min(),
        "F1 máximo": tabla_arquitectura["F1 macro promedio"].max()
    },
    {
        "Hiperparámetro": "Regularización alpha",
        "F1 mínimo": tabla_alpha_s5["F1 macro promedio"].min(),
        "F1 máximo": tabla_alpha_s5["F1 macro promedio"].max()
    },
    {
        "Hiperparámetro": "Tasa de aprendizaje",
        "F1 mínimo": tabla_lr_s5["F1 macro promedio"].min(),
        "F1 máximo": tabla_lr_s5["F1 macro promedio"].max()
    },
    {
        "Hiperparámetro": "Características TF-IDF",
        "F1 mínimo": tabla_features_s5["F1 macro promedio"].min(),
        "F1 máximo": tabla_features_s5["F1 macro promedio"].max()
    }
])

resumen_sensibilidad_s5["Variación F1"] = (
    resumen_sensibilidad_s5["F1 máximo"]
    - resumen_sensibilidad_s5["F1 mínimo"]
)

display(
    resumen_sensibilidad_s5.sort_values(
        "Variación F1",
        ascending=False
    ).round(4)
)

In [ ]:
import matplotlib.pyplot as plt

tabla_grafico = resumen_sensibilidad_s5.sort_values(
    "Variación F1",
    ascending=True
)

plt.figure(figsize=(10, 5))

barras = plt.barh(
    tabla_grafico["Hiperparámetro"],
    tabla_grafico["Variación F1"],
    color="steelblue"
)

plt.bar_label(barras, fmt="%.4f", padding=4)

plt.xlabel("Variación observada del F1 macro")
plt.ylabel("Hiperparámetro")
plt.title("Semana 5 - Sensibilidad de hiperparámetros")
plt.xlim(0, 0.085)
plt.tight_layout()

plt.savefig(
    "sensibilidad_hiperparametros_s5.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from itertools import product

resultados_interaccion_s5 = []

for arquitectura, lr in product(
    [(64, 32), (128, 64)],
    [0.001, 0.01]
):
    modelo_interaccion = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=1000,
            min_df=2,
            ngram_range=(1, 2)
        )),
        ("modelo", MLPClassifier(
            hidden_layer_sizes=arquitectura,
            alpha=0.0001,
            learning_rate_init=lr,
            max_iter=400,
            random_state=42
        ))
    ])

    scores = cross_validate(
        modelo_interaccion,
        textos_s5,
        etiquetas_s5,
        groups=grupos_s5,
        cv=cv_s5,
        scoring={
            "f1_macro": "f1_macro",
            "recall_positivo": "recall"
        },
        n_jobs=1
    )

    resultados_interaccion_s5.append({
        "Arquitectura": str(arquitectura),
        "Learning rate": lr,
        "F1 macro promedio": np.mean(scores["test_f1_macro"]),
        "Recall positivo promedio": np.mean(
            scores["test_recall_positivo"]
        )
    })

tabla_interaccion_s5 = pd.DataFrame(
    resultados_interaccion_s5
)

display(tabla_interaccion_s5.round(4))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

for arquitectura in tabla_interaccion_s5["Arquitectura"].unique():
    datos = tabla_interaccion_s5[
        tabla_interaccion_s5["Arquitectura"] == arquitectura
    ].sort_values("Learning rate")

    plt.plot(
        datos["Learning rate"].astype(str),
        datos["F1 macro promedio"],
        marker="o",
        linewidth=2,
        markersize=8,
        label=f"MLP {arquitectura}"
    )

plt.xlabel("Tasa de aprendizaje")
plt.ylabel("F1 macro promedio")
plt.title("Semana 5 - Interacción entre arquitectura y tasa de aprendizaje")
plt.ylim(0.89, 0.93)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "interaccion_arquitectura_learning_rate_s5.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

carpeta_evidencias = (
    '/content/drive/MyDrive/'
    'UEES_Proyecto_Integrador/'
    'Semana5_Resultados'
)

os.makedirs(carpeta_evidencias, exist_ok=True)

archivos = [
    'sensibilidad_hiperparametros_s5.png',
    'interaccion_arquitectura_learning_rate_s5.png'
]

for archivo in archivos:
    if os.path.exists(archivo):
        shutil.copy2(
            archivo,
            os.path.join(carpeta_evidencias, archivo)
        )
        print('Guardado:', archivo)
    else:
        print('No encontrado:', archivo)

print('\nCarpeta:', carpeta_evidencias)

## 9. Resultados y conclusiones de la optimización

Se realizó un análisis de sensibilidad del clasificador TF-IDF + MLP utilizando 201 registros de entrenamiento y validación cruzada agrupada de cinco particiones. Se mantuvieron separados los 71 registros destinados a la evaluación final.

Se evaluaron cuatro hiperparámetros: arquitectura de la red neuronal, regularización (`alpha`), tasa de aprendizaje (`learning_rate_init`) y número máximo de características TF-IDF (`max_features`).

### Principales resultados

- **Arquitectura:** la configuración `(128, 64)` alcanzó un F1 macro de 0,9127, frente a 0,9125 de la arquitectura original `(64, 32)`. La diferencia fue mínima, por lo que se prefirió conservar la arquitectura original por su menor complejidad.
- **Regularización:** los valores `alpha=0.0001` y `alpha=0.001` obtuvieron un F1 macro de 0,9125. Una regularización más fuerte redujo el rendimiento, llegando a 0,8790 con `alpha=1.0`.
- **Tasa de aprendizaje:** el valor `0.001` obtuvo un F1 macro de 0,9125. El valor `0.01` alcanzó 0,9099 y un mayor recall de la clase positiva (0,8483 frente a 0,8067). El valor `0.0001` no alcanzó la convergencia dentro de las 400 iteraciones, por lo que su resultado debe interpretarse con precaución.
- **Características TF-IDF:** utilizar 1.000 características máximas permitió alcanzar un F1 macro de 0,9125. Aumentar el límite a 2.000 o 3.000 no produjo mejoras, ya que el vocabulario completo disponible contenía 828 características.

### Interacción entre hiperparámetros

Se analizaron combinaciones de dos arquitecturas y dos tasas de aprendizaje. Con una tasa de `0.001`, ambas arquitecturas obtuvieron resultados prácticamente iguales. Con una tasa de `0.01`, la arquitectura `(128, 64)` presentó una reducción mayor del F1 macro.

Los resultados sugieren que el efecto de la tasa de aprendizaje puede depender de la arquitectura, aunque no se realizaron pruebas estadísticas para determinar la significancia de estas diferencias.

### Configuración seleccionada provisionalmente

Se conserva la arquitectura `(64, 32)`, con `alpha=0.0001`, `learning_rate_init=0.001`, `max_features=1000` y `max_iter=400`.

Esta selección se fundamenta en el equilibrio entre rendimiento observado, estabilidad y complejidad del modelo.

### Limitaciones

Los resultados corresponden a validación cruzada sobre los datos de entrenamiento. No constituyen una evaluación independiente del modelo ni demuestran capacidad de diagnóstico clínico. Algunos experimentos presentaron advertencias de convergencia y la muestra disponible es limitada.

La evaluación final sobre los 71 registros reservados deberá analizarse por separado, sin utilizarlos para seleccionar hiperparámetros.

In [ ]:
print("Registros reservados:", len(val))
print("Etiquetas reservadas:", len(y_val))
print("Dimensiones de X_val:", X_val.shape)

print("\nColumnas disponibles:")
print(val.columns.tolist())

print("\nDistribución de etiquetas:")
print(pd.Series(y_val).value_counts().sort_index())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

modelo_final_s5 = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=1000,
        min_df=2,
        ngram_range=(1, 2)
    )),
    ("modelo", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        alpha=0.0001,
        learning_rate_init=0.001,
        max_iter=400,
        random_state=42
    ))
])

# Entrenar únicamente con los 201 registros
modelo_final_s5.fit(textos_s5, etiquetas_s5)

# Evaluar una sola vez sobre los 71 registros reservados
textos_evaluacion_s5 = (
    val["texto_sin_versos"].fillna("").astype(str)
)

predicciones_final_s5 = modelo_final_s5.predict(
    textos_evaluacion_s5
)

print("Accuracy:",
      round(accuracy_score(y_val, predicciones_final_s5), 4))

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_val, predicciones_final_s5), 4))

print("F1 macro:",
      round(f1_score(y_val, predicciones_final_s5, average="macro"), 4))

print("Recall clase positiva:",
      round(recall_score(y_val, predicciones_final_s5), 4))

print("\nReporte de clasificación:")
print(classification_report(
    y_val,
    predicciones_final_s5,
    zero_division=0
))

print("\nMatriz de confusión:")
print(confusion_matrix(y_val, predicciones_final_s5))

In [ ]:
pred_train_s5 = modelo_final_s5.predict(textos_s5)

print("F1 macro entrenamiento:",
      round(f1_score(
          etiquetas_s5,
          pred_train_s5,
          average="macro"
      ), 4))

print("F1 macro evaluación final:",
      round(f1_score(
          y_val,
          predicciones_final_s5,
          average="macro"
      ), 4))

In [ ]:
longitudes_train = pd.Series(textos_s5).str.split().str.len()

longitudes_val = (
    val["texto_sin_versos"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

comparacion_textos_s5 = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Evaluación final"],
    "Registros": [len(longitudes_train), len(longitudes_val)],
    "Promedio de palabras": [
        longitudes_train.mean(),
        longitudes_val.mean()
    ],
    "Mediana de palabras": [
        longitudes_train.median(),
        longitudes_val.median()
    ],
    "Máximo de palabras": [
        longitudes_train.max(),
        longitudes_val.max()
    ]
})

display(comparacion_textos_s5.round(2))

In [ ]:
comparacion_clases_s5 = pd.DataFrame({
    "Entrenamiento (%)": (
        pd.Series(etiquetas_s5)
        .value_counts(normalize=True)
        .sort_index() * 100
    ),
    "Evaluación final (%)": (
        pd.Series(y_val)
        .value_counts(normalize=True)
        .sort_index() * 100
    )
})

comparacion_clases_s5.index.name = "Clase"

display(comparacion_clases_s5.round(2))

In [ ]:
vectorizador_s5 = modelo_final_s5.named_steps["tfidf"]

matriz_val_s5 = vectorizador_s5.transform(
    textos_evaluacion_s5
)

documentos_sin_vocabulario = (
    matriz_val_s5.getnnz(axis=1) == 0
).sum()

print("Características aprendidas:",
      len(vectorizador_s5.vocabulary_))

print("Informes sin características reconocidas:",
      documentos_sin_vocabulario)

print("Promedio de características reconocidas por informe:",
      round(matriz_val_s5.getnnz(axis=1).mean(), 2))

In [ ]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(
    y_val,
    predicciones_final_s5
).ravel()

print("Verdaderos negativos:", tn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdaderos positivos:", tp)

print("\nTasa de falsos negativos:",
      f"{fn / (fn + tp) * 100:.2f}%")

print("Tasa de falsos positivos:",
      f"{fp / (fp + tn) * 100:.2f}%")

In [ ]:
import json
import os

resultados_finales_s5 = {
    "registros_entrenamiento": 201,
    "registros_evaluacion": 71,
    "f1_macro_entrenamiento": 0.9722,
    "f1_macro_validacion_cruzada": 0.9125,
    "accuracy_evaluacion": 0.8169,
    "balanced_accuracy_evaluacion": 0.7386,
    "f1_macro_evaluacion": 0.7434,
    "recall_clase_positiva": 0.5882,
    "verdaderos_negativos": 48,
    "falsos_positivos": 6,
    "falsos_negativos": 7,
    "verdaderos_positivos": 10,
    "tasa_falsos_negativos": 0.4118,
    "tasa_falsos_positivos": 0.1111
}

ruta_resultados = os.path.join(
    carpeta_evidencias,
    "metricas_evaluacion_final_s5.json"
)

with open(ruta_resultados, "w", encoding="utf-8") as archivo:
    json.dump(
        resultados_finales_s5,
        archivo,
        indent=4,
        ensure_ascii=False
    )

print("Resultados guardados en:")
print(ruta_resultados)

## 10. Evaluación final y análisis de generalización

### Evaluación sobre datos reservados

Después de seleccionar los hiperparámetros mediante validación cruzada agrupada, se entrenó el modelo TF-IDF + MLP con los 201 registros de entrenamiento y se evaluó sobre 71 estudios reservados.

Los resultados fueron:

| Métrica | Resultado |
|---|---:|
| Accuracy | 0,8169 |
| Balanced accuracy | 0,7386 |
| F1 macro | 0,7434 |
| Recall de la clase positiva | 0,5882 |

La matriz de confusión registró 48 verdaderos negativos, 10 verdaderos positivos, 6 falsos positivos y 7 falsos negativos.

La tasa de falsos negativos fue del 41,18 %, mientras que la de falsos positivos fue del 11,11 %. Esto evidencia una limitación importante en la identificación de informes que contienen versos configurados.

### Análisis de posible sobreajuste

El F1 macro obtenido en entrenamiento fue de 0,9722, mientras que en validación cruzada fue de 0,9125 y en evaluación final de 0,7434.

La diferencia de 22,88 puntos porcentuales entre entrenamiento y evaluación final es compatible con un problema de sobreajuste o de generalización. Sin embargo, estos resultados no permiten determinar una causa única.

Se realizaron comprobaciones descriptivas adicionales:

- La proporción de la clase positiva fue del 22,39 % en entrenamiento y del 23,94 % en evaluación.
- La longitud promedio fue de 63,91 palabras en entrenamiento y 63,99 en evaluación.
- Ninguno de los 71 informes reservados quedó sin características reconocidas por el vectorizador TF-IDF.

Por lo tanto, no se observaron diferencias importantes en la distribución de clases ni en la longitud promedio de los textos. No obstante, podrían existir diferencias de vocabulario, contenido o patrones de redacción que requieran un análisis posterior.

### Limitaciones y trabajo futuro

El conjunto de datos es reducido y presenta desbalance entre clases. Los resultados obtenidos no son suficientes para utilizar el clasificador de manera autónoma en un entorno clínico.

Como trabajo futuro se propone ampliar y diversificar el conjunto de datos, estudiar los errores de clasificación y evaluar nuevas estrategias de entrenamiento mediante conjuntos de validación independientes.

La evaluación final se mantuvo separada del proceso de selección de hiperparámetros para evitar utilizar los datos reservados en la optimización.

**Conclusión:** el clasificador constituye un componente experimental de apoyo para identificar la presencia de versos configurados en informes radiológicos, pero requiere mejoras de generalización. La selección y aprobación del informe deben permanecer bajo supervisión humana.

In [ ]:
import matplotlib.pyplot as plt

print("Columnas disponibles:")
print(tabla_alpha_s5.columns.tolist())

display(tabla_alpha_s5)

In [ ]:
import matplotlib.pyplot as plt
import os

fig, ax = plt.subplots(figsize=(9, 5))

ax.errorbar(
    tabla_alpha_s5["Alpha"],
    tabla_alpha_s5["F1 macro promedio"],
    yerr=tabla_alpha_s5["F1 macro desviación"],
    marker="o",
    capsize=4,
    linewidth=2,
    label="F1 macro (media ± desviación)"
)

ax.set_xscale("log")
ax.set_xlabel("Regularización (alpha)")
ax.set_ylabel("F1 macro")
ax.set_title("Sensibilidad del modelo a la regularización")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()

plt.tight_layout()

ruta_grafica_alpha = os.path.join(
    carpeta_evidencias,
    "sensibilidad_alpha_s5.png"
)

plt.savefig(ruta_grafica_alpha, dpi=200, bbox_inches="tight")
plt.show()

print("Gráfica guardada en:", ruta_grafica_alpha)

In [ ]:
print("Columnas disponibles:")
print(tabla_lr_s5.columns.tolist())

display(tabla_lr_s5)

In [ ]:
import matplotlib.pyplot as plt
import os

fig, ax = plt.subplots(figsize=(9, 5))

ax.errorbar(
    tabla_lr_s5["Learning rate"],
    tabla_lr_s5["F1 macro promedio"],
    yerr=tabla_lr_s5["F1 macro desviación"],
    marker="o",
    linewidth=2,
    capsize=5,
    label="F1 macro (media ± desviación)"
)

ax.set_xscale("log")
ax.set_xlabel("Tasa de aprendizaje (learning_rate_init)")
ax.set_ylabel("F1 macro")
ax.set_title("Sensibilidad del modelo a la tasa de aprendizaje")
ax.grid(True, linestyle="--", alpha=0.4)

# Identificar el experimento que alcanzó el límite de iteraciones
for _, fila in tabla_lr_s5.iterrows():
    if fila["Folds en límite de iteraciones"] > 0:
        ax.annotate(
            "5/5 folds alcanzaron 400 iteraciones",
            (fila["Learning rate"], fila["F1 macro promedio"]),
            xytext=(15, -35),
            textcoords="offset points",
            fontsize=9,
            arrowprops={"arrowstyle": "->"}
        )

ax.legend()
plt.tight_layout()

ruta_grafica_lr = os.path.join(
    carpeta_evidencias,
    "sensibilidad_learning_rate_s5.png"
)

plt.savefig(ruta_grafica_lr, dpi=200, bbox_inches="tight")
plt.show()

print("Gráfica guardada en:", ruta_grafica_lr)

In [ ]:
print("Columnas disponibles:")
print(tabla_arquitectura.columns.tolist())

display(tabla_arquitectura)

In [ ]:
import matplotlib.pyplot as plt
import os

arquitecturas = tabla_arquitectura["Arquitectura"].astype(str)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    arquitecturas,
    tabla_arquitectura["F1 macro promedio"],
    yerr=tabla_arquitectura["F1 macro desviación"],
    capsize=6
)

ax.set_xlabel("Arquitectura MLP")
ax.set_ylabel("F1 macro promedio")
ax.set_title("Sensibilidad del modelo a la arquitectura MLP")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.4)

for i, valor in enumerate(tabla_arquitectura["F1 macro promedio"]):
    ax.text(
        i, valor + 0.07,
        f"{valor:.4f}",
        ha="center"
    )

plt.tight_layout()

ruta_grafica_arquitectura = os.path.join(
    carpeta_evidencias,
    "sensibilidad_arquitectura_s5.png"
)

plt.savefig(
    ruta_grafica_arquitectura,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Gráfica guardada en:", ruta_grafica_arquitectura)

In [ ]:
print("Columnas disponibles:")
print(tabla_features_s5.columns.tolist())

display(tabla_features_s5)

In [ ]:
import matplotlib.pyplot as plt
import os

fig, ax = plt.subplots(figsize=(9, 5))

ax.errorbar(
    tabla_features_s5["Max features"],
    tabla_features_s5["F1 macro promedio"],
    yerr=tabla_features_s5["F1 macro desviación"],
    marker="o",
    linewidth=2,
    capsize=5,
    label="F1 macro (media ± desviación)"
)

ax.set_xlabel("Número máximo de características TF-IDF")
ax.set_ylabel("F1 macro")
ax.set_title("Sensibilidad del modelo a max_features")
ax.set_xticks(tabla_features_s5["Max features"])
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()

for _, fila in tabla_features_s5.iterrows():
    ax.annotate(
        f'{fila["F1 macro promedio"]:.4f}',
        (fila["Max features"], fila["F1 macro promedio"]),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

ruta_grafica_features = os.path.join(
    carpeta_evidencias,
    "sensibilidad_tfidf_features_s5.png"
)

plt.savefig(
    ruta_grafica_features,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Gráfica guardada en:", ruta_grafica_features)

In [ ]:
import os

tablas_s5 = {
    "resultados_arquitectura_s5.csv": tabla_arquitectura,
    "resultados_regularizacion_s5.csv": tabla_alpha_s5,
    "resultados_learning_rate_s5.csv": tabla_lr_s5,
    "resultados_tfidf_s5.csv": tabla_features_s5,
    "resultados_interaccion_s5.csv": tabla_interaccion_s5,
    "ranking_sensibilidad_s5.csv": resumen_sensibilidad_s5
}

for nombre_archivo, tabla in tablas_s5.items():
    ruta = os.path.join(carpeta_evidencias, nombre_archivo)

    tabla.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print("Guardado:", nombre_archivo)

print("\nTotal de tablas guardadas:", len(tablas_s5))

In [ ]:
import os

archivos = sorted(os.listdir(carpeta_evidencias))

print("EVIDENCIAS DE LA SEMANA 5")
print("=" * 45)

for archivo in archivos:
    print("✓", archivo)

print("\nTotal de archivos:", len(archivos))

In [ ]:
print("Modelo:", type(modelo_final_s5.named_steps["modelo"]).__name__)

print(
    "Permite calcular probabilidades:",
    hasattr(modelo_final_s5, "predict_proba")
)

print(
    "Número de características TF-IDF:",
    len(modelo_final_s5.named_steps["tfidf"].get_feature_names_out())
)

In [ ]:
import numpy as np
import pandas as pd

vectorizador_s5 = modelo_final_s5.named_steps["tfidf"]

matriz_tfidf_s5 = vectorizador_s5.transform(textos_s5)

terminos_s5 = vectorizador_s5.get_feature_names_out()

promedios_s5 = np.asarray(
    matriz_tfidf_s5.mean(axis=0)
).ravel()

tabla_terminos_s5 = pd.DataFrame({
    "Término": terminos_s5,
    "TF-IDF promedio": promedios_s5
})

tabla_terminos_s5 = (
    tabla_terminos_s5
    .sort_values("TF-IDF promedio", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

display(tabla_terminos_s5)

In [ ]:
terminos_candidatos = [
    "imágenes",
    "mano",
    "vertebrales",
    "posición",
    "fractura",
    "normal",
    "lesión",
    "no hay"
]

vocabulario = vectorizador_s5.vocabulary_

resultados = []

for termino in terminos_candidatos:
    if termino in vocabulario:
        indice = vocabulario[termino]

        frecuencia = (
            matriz_tfidf_s5[:, indice].toarray().ravel() > 0
        ).sum()

        resultados.append({
            "Término": termino,
            "Disponible": "Sí",
            "Informes que lo contienen": int(frecuencia)
        })
    else:
        resultados.append({
            "Término": termino,
            "Disponible": "No",
            "Informes que lo contienen": 0
        })

display(pd.DataFrame(resultados))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Característica que analizaremos
termino_pdp = "fractura"
indice_pdp = vocabulario[termino_pdp]

# Representación TF-IDF de los datos de entrenamiento
X_pdp = matriz_tfidf_s5.toarray()

# Valores observados de la característica
valores_observados = X_pdp[:, indice_pdp]

# Evaluamos valores dentro del rango observado
valores_prueba = np.linspace(
    valores_observados.min(),
    valores_observados.max(),
    20
)

# Modelo MLP ya entrenado
mlp_pdp = modelo_final_s5.named_steps["modelo"]

probabilidades_promedio = []

for valor in valores_prueba:
    X_modificado = X_pdp.copy()

    # Modificamos únicamente la característica seleccionada
    X_modificado[:, indice_pdp] = valor

    probabilidades = mlp_pdp.predict_proba(
        X_modificado
    )[:, 1]

    probabilidades_promedio.append(
        probabilidades.mean()
    )

# Gráfica
plt.figure(figsize=(9, 5))

plt.plot(
    valores_prueba,
    probabilidades_promedio,
    marker="o"
)

plt.xlabel("Valor TF-IDF del término 'fractura'")
plt.ylabel("Probabilidad promedio estimada de clase 1")
plt.title("Dependencia parcial exploratoria: fractura")
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()

ruta_pdp = os.path.join(
    carpeta_evidencias,
    "dependencia_parcial_fractura_s5.png"
)

plt.savefig(ruta_pdp, dpi=200, bbox_inches="tight")
plt.show()

print("Gráfica guardada en:", ruta_pdp)

In [ ]:
termino_pdp_2 = "vertebrales"
indice_pdp_2 = vocabulario[termino_pdp_2]

valores_observados_2 = X_pdp[:, indice_pdp_2]

valores_prueba_2 = np.linspace(
    valores_observados_2.min(),
    valores_observados_2.max(),
    20
)

probabilidades_promedio_2 = []

for valor in valores_prueba_2:
    X_modificado = X_pdp.copy()
    X_modificado[:, indice_pdp_2] = valor

    probabilidades = mlp_pdp.predict_proba(
        X_modificado
    )[:, 1]

    probabilidades_promedio_2.append(
        probabilidades.mean()
    )

plt.figure(figsize=(9, 5))

plt.plot(
    valores_prueba_2,
    probabilidades_promedio_2,
    marker="o"
)

plt.xlabel("Valor TF-IDF del término 'vertebrales'")
plt.ylabel("Probabilidad promedio estimada de clase 1")
plt.title("Dependencia parcial exploratoria: vertebrales")
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()

ruta_pdp_2 = os.path.join(
    carpeta_evidencias,
    "dependencia_parcial_vertebrales_s5.png"
)

plt.savefig(ruta_pdp_2, dpi=200, bbox_inches="tight")
plt.show()

print("Gráfica guardada en:", ruta_pdp_2)

In [ ]:
import pandas as pd
import os

tabla_pdp_fractura = pd.DataFrame({
    "TF-IDF": valores_prueba,
    "Probabilidad promedio clase 1": probabilidades_promedio
})

tabla_pdp_vertebrales = pd.DataFrame({
    "TF-IDF": valores_prueba_2,
    "Probabilidad promedio clase 1": probabilidades_promedio_2
})

tabla_pdp_fractura.to_csv(
    os.path.join(carpeta_evidencias, "dependencia_parcial_fractura_s5.csv"),
    index=False
)

tabla_pdp_vertebrales.to_csv(
    os.path.join(carpeta_evidencias, "dependencia_parcial_vertebrales_s5.csv"),
    index=False
)

print("Guardados los dos archivos CSV de dependencia parcial.")
print("\nResumen de fractura:")
display(tabla_pdp_fractura.iloc[[0, -1]])

print("\nResumen de vertebrales:")
display(tabla_pdp_vertebrales.iloc[[0, -1]])

## 11. Análisis exploratorio de dependencia parcial

Como complemento al análisis de sensibilidad de hiperparámetros, se evaluó cómo cambia la probabilidad promedio estimada para la clase positiva al modificar individualmente dos características TF-IDF: «fractura» y «vertebrales».

Para cada término se modificó su valor TF-IDF dentro del rango observado en los datos de entrenamiento, manteniendo constantes las demás características. Se utilizó el clasificador MLP previamente entrenado, sin realizar un nuevo entrenamiento.

**Resultados obtenidos:**

- **Fractura:** la probabilidad promedio estimada pasó de 0,241772 a 0,239597, una disminución aproximada de 0,22 puntos porcentuales.
- **Vertebrales:** la probabilidad promedio estimada pasó de 0,241870 a 0,239967, una disminución aproximada de 0,19 puntos porcentuales.

Los resultados muestran variaciones pequeñas en la probabilidad promedio de la clase positiva. Esto sugiere que, dentro de los rangos analizados, modificar aisladamente estas características tiene un efecto promedio limitado sobre las predicciones del modelo.

**Limitaciones:** este análisis es exploratorio. La modificación individual de características TF-IDF puede generar combinaciones que no correspondan a textos reales. Además, las curvas representan efectos promedio y pueden ocultar diferencias entre registros o interacciones entre términos. Por tanto, los resultados no deben interpretarse como relaciones causales ni como explicaciones clínicas.

El objetivo del clasificador es estimar si un informe contiene un verso configurado, no identificar enfermedades ni establecer diagnósticos radiológicos.

Las gráficas y los valores utilizados se conservaron como evidencias para permitir su revisión y reproducción.